## Architecture tổng thể

OFFLINE PIPELINE (chạy định kỳ / khi có tài liệu mới)

1.Documents (10k support docs + 500 FAQ + 2K tickets)
    
2.Ingestion   -   đọc từ nguồn (CMS, ticket system, FAQ DB), chuẩn hóa format
    
3.Document parsing    - trích text sạch từ HTML/PDF/markdown, loại bỏ noise
    
4.Chunking    -   cắt tài liệu dài thành đoạn nhỏ vừa embed
    
5.Metadata extraction -   gắn document_type, product, version, created_at...
    
6.Embedding   -   biến mỗi chunk thành vector số
    
7.Indexing    -   lưu vector vào Vector DB + lưu text vào BM25 index


ONLINE QUERY PIPELINE (chạy mỗi lần user hỏi)

1.USER QUERY
    
2.Query Processing    -   chuẩn hóa, có thể review/expand query

3.Retrieval -   vector search + BM25 (Experiment A-D)

4.Reranking    -   sắp xếp lại candidate theo độ liên quan chính xác hơn

5.Context Construction  -   chọn+sắp xếp chunk vào context, tuân thủ token budget

6. LLM  -   sinh câu trả lời dựa trên context

7. Answer

Vai trò từng component

Ingestion: đầu vào thống nhất từ 3 nguồn khác định dạng - đây là nơi cần chuẩn hóa schema chung trước khi các bước sau xử lý đồng nhất

Parsing: tách nội dung có ý nghĩa khỏi format gốc 

Chunking: quyết định đơn vị tìm kiếm - quá lớn thì lẫn ý nghĩa, quá nhỏ thì mất ngữ cảnh

Metadata extraction: tạo ra trục lọc phụ (không dựa vào semantic similarity) - quan trọng cho access control và độ chính xác

Embedding: biến text -> vector để só sánh semantic similarity

Indexing: cấu trúc dữ liệu cho phép tìm kiếm nhanh

Query Processing: query nguời dùng viết thường khác cách viết trong tài liệu - bước này thu hẹp khoảng cách đó

Retrieval: lấy tập ứng viên ban đầu (rẻ, nhanh, recall cao)

Reranking: lọc lại ứng viên cho chính xác hơn (đắt nhưng áp dụng trên tập nhỏ)

Context Construction: không phải nhét hết candidate vào - cần chọn lọc theo budget token

LLM: tổng hợp câu trả lời cuối - grounded trên context đã cung cấp

Offline vs Online - vì sao tách biệt quan trọng? Offline pipeline chạy 1 lần (dịnh kỳ) cho toàn bộ knowledge base, có thể chậm/tốn tài nguyên vì không ảnh hưởng đến trải nghiệm user thời gian thực. Online pipeline phải nhanh (mỗi query có latency budget), nên mọi phần tính toán nặng (embed toàn bộ documents, xây index) phải đẩ y hết sang offline - chỉ embed query 1 đoạn ngắn ở online


    

## Quyết định từng parameter

### Chunk size
Baseline đề xuất: 512 token, overlap 15%
Lý do: Support docs/FAQ thường có cấu trúc "1 vấn đề = 1 đoạn giải thích" tương đối ngắn gọn - 512 token đủ chứa trọng 1 ý hoàn chỉnh 

Trade-off
- Chunk nhỏ (256): tăng độ chính xác định vị, nhưng dễ mất ngữ cảnh, tăng số lượng chunk cần index (chi phí+storage+tìm kiếm)
- Chunk lớn (1024): giữ ngữ cảnh tốt hơn, nhưng embedding của 1 chunk dài dễ bị trung bình hóa ý nghĩa (nếu chunk có chứa nhiều chủ đề, vector đại diện kém cho từng ý riêng), giảm độ chính xác retrieval.

Khi nào baseline thất bại: Với ticket đã giải quết - nội dung thường dài, nhiều lượt trao đổi qua lại, 512 token có thể cắt giữa 1 luồng hội thoại quan trọng -> có thể cần chunking strategy khác cho loại tài liệu này. VD: chunk theo từng lượt trao đổi, không theo token cố định.

Cách experiement tối ưu: Chạy thử 256/512/768/1024 trên cùng eval set, đo Recall@K - chunk size tối ưu là điểm mà Recall@K không còn tăng đáng kể khi tăng size thêm 

### Chunk overlap
Baseline: 15% (~75 token với chunk với chunk512)

Lý do: Overlap tránh việc 1 câu quan trọng bị cắt đúng vào ranh giới 2 chunk, khiến cả 2 chunk đều mất phần ý nghĩa đó.

Quá ít (0-5%): Nguy cơ cao câu/ ý nghĩa bị cắt đứt giữa chừng, retrieval có thể miss chunk chứa nửa câu trả lời

Quá nhiều (40%): Trùng lặp dữ liệu index (tốn storage), retrieval trả về nhiều chunk gần giống nhau (giảm diversity của candidate set), tăng nhiễu trong context construction.

### Embedding model
Tiêu chí chọn:
- Multilingual quan trọng nếu knowledge base + query có cả Việt và Anh - model chỉ train tiếng anh sẽ embed câu tiếng Việt kém chính xác, giảm recall cho toàn bộ nhóm query tiếng Việt.
- Vietnamese query support: cần kiểm tra cụ thể model có benchmark tốn trên tiếng Việt không - đây là điểm bắt buộc phải test thực tế, không đoán.
- Embedding dimension: dimension cao hơn thường biểu diễn semantic toonshonw nhưng tốn storage + latency tìm kiếm hơn (vector DB phải so sánh vector dài hơn); dimension thấp nhanh/rẻ hơn có thể mất sắc thái ngữ nghĩa tinh tế.
- Model size ảnh hưởng latency/cost: model lớn hơn -> embed chậm hơn (ảnh hưởng cả offline ingestion lẫn online embed-query-mỗi lần hỏi) và thường tính phí theo token cao hơn nếu dùng API thay vì self host.

### Top-k
Nguyễn tắc: Top k ở bước retrieval nên rộng hơn số chunk cuối cùng đưa vào LLM - vì retrieval (đặc biệt vector search) tối ưu cho recall cao, tốc độ nhanh, không phải độ chính xác tôi đa. Baseline đề xuất: retrieve top-20-30, rerank xuống còn top-5

Tại sao không lây top-3 luôn từ đầu? Vì embedding similarity không hoàn hảo - chunk thực sự liên quan nhất đôi khi xếp hạng 7-8 trong vector search thô, những sẽ được reranker (chính xác hơn nhưng đắt hơn) đưa lên đúng vị trị. Lấy top-3 ngay từ bước retrieval rẻ có nguy cơ loại bỏ oan chunk đúng trước khi nó có cơ hội được đánh giá lại.

### Reranker
Reranker khác retrieval như thế nào? Retrieval (vector/BM25) tính độ liên quan độc lập giữa query và mỗi document (bi-encoder - embed query và document riêng biệt rồi s sánh vector) -> nhanh nhưng kém chính xác. Reranker thường dùng cross-encoder - đưa cả query và document vào cùng 1 lần qua model để so sánh trực tiếp, tính tương tác giauwx 2 văn bản -> chính hơn nhiều nhưng chậm hơn
(không thể tính trước, phải chạy real-time cho từng cặp).

Tại sao không dùng reranker cho toàn bộ 12,500 tài liệu? Vì cross encoder phải chạy cho từng cặp (query, document) riêng lẻ - với 12,500 document, mỗi query sẽ cần 12,500 lần inference, quá chậm cho real-time. Đây là lý do kiến trúc 2 tầng: retrieval rẻ trước để thu hẹp xuống vài chục candidate, reranker đắt sau chỉ chạy trên tập nhỏ đó.

Candidate set nên bao nhiêu? Baseline 20-30 đủ rộng để bắt được chunk đúng dù retrieval thô xếp hạng chưa chuẩn, nhưng đủ nhỏ đê reranker chạy nhanh

### Metadata filer

Schema đề xuất: 

document_id, document_type(doc/faq/ticket), product, version, language, category, created_at, updated_at, customer_tier,
source, access_level

Cải thiện precision thế nào? 

Metadata filter loại bỏ candidate trước khi tính semantic similarity, dựa trên điều kiện chắc chắn đúng sai (không mờ như similarity score).

### Context limit

Tại sao không đưa tất cả retrieved chunk vào LLM?

Ngay cả sau reranking, không phải mọi chunk trong top-N đều thực sự cần thiết - nhồi hết vẫn gặp lại vấn đề lost in the middle và tăng cost.

Quá dài: loãng tín hiệu quan trọng, tăng cost, tăng latency, tăng nguy cơ context chưa thông tin mâu thuẫn nhau.

Quá ngắn: thiếu ngữ cảnh cần thiết để trả lời đầy đủ, đặc biệt câu hỏi cần tổng hợp từ nhiều nguồn

Phân bố context budget: Baseline - dành ~70% budget cho top chunk theo rerank score, ~20% cho metadata context, ~10% buffer cho system prompt/instruction



Thiết kế 4 experimennt
Experiment A - Vector Search
Ưu điểm: Bắt được semantic similarity - hiểu ý nghĩa dù từ ngữ khác nhau 
Nhược điểm: Yếu với exact match - mã lỗi cụ thể, tên sản phẩm chính xác, ticket ID - vì embedding làm mờ chi tiết chính xác thành biểu diễn ngữ nghĩa tổng quát.
Phù hợp: Câu hỏi diễn đạt tự nhiên, đồng nghĩa, diễn giải khác cách viết trong tài liệu.
Dễ fail: Query chứa mã lỗi/số phiên bản cụ thể - vector search có thể trả về chunk nói về lỗi khác có ngữ cảnh giống giống nhưng sai mã cụ thể.

Experiment B - BM25
Tại sao vẫn hữu ích trong thời đại embedding? BM25 (thuật toán tính điểm dựa trên tần suất từ khóa - term frequency) không làm mờ chi tiết chính xác - nó khớp từ khóa nguyên văn, nên với các loại query sau, BM25 thường vượt trội hơn vector search.
- Product name cụ thể
- Error code
- Ticket ID
- Version number
- Technical keyword
- Exact phrase

Đây là các trường hợp mà nghĩa không quan trọng bằng khớp chuỗi kí tự chính xác

Experiement C - Hybrid Search
Query -> vector Search (semantic)
      -> BM 25 (lexical)
            |
            Fusion
            |Top-k

Score fusion vs weighted fusion: Score fusion đơn giản cộng/nhân trực tiếp 2 điểm số - nhưng vector similarity (thường 0-1, cosine) và BM25 score (không giới hạn, phụ thuộc corpus) có thang đo khac nhau hoàn toàn, cộng trực tiếp sẽ bị lệch về phía điểm có magnitude lớn hơn. Weighted fusion cố gắng cân bằng hệ số nhân, nhưng vẫn cần chuản hóa 2 thang đo trước.

Reciporal Rank Fusion (RRF): THay vì dùng điểm số thô - RRF chỉ dùng thứ hạng (rank) của document tỏng mỗi danh sách kết quả score = sigma 1/(k+rank) với k là hằng số. Ưu điểm không cần chuẩn hóa thang điểm giữa 2 phương pháp khác nhau - đây là lý do RRF thường được chọn làm baseline fusion vì đơn giản và ổn định hơn weight score function khi chưa có dữ liệu để tune trọng số.

Baseline đề xuất: RRF với k = 60 (giá trị phổ biến trong literature, là điểm khởi đầu hợp lý - không phải optimum, cần eval thực tế để tinh chỉnh)

Experiment D: Hybrid + Reranker
Queery -> BM25  ----- 
                    | -> Candiate Pool -> Reranker -> Top-N context -> LLM
        vector ------
Tại sao có thể tốt nhất nhưng không phải nhất thiết luôn tốt nhất? Vì tốt nhất phụ thuộc vào mục tiêu tối ưu - nếu chỉ nhìn quality, đúng là hyrbrid +reranker thường thắng vì kết hợp cả recall rộng (hybrid) lẫn precision cao (reranker). Nhưng nhìn latency : đây là pipeline có nhiều bước tuần tự nhất (2 lần retrieval + fusion _+ reranker) -> chậm nhất trong 4 phương án. Nhìn Cost: reranker (đặc biệt cross-encoder lơn) thường đắt hơn embedding thuần túy. Với use case cần phản hồi tức thời (VD: live chat support), độ trễ thêm từ reranker có thể không đáng đánh đổi nếu Hybrid đơn thuần đã đủ chất lượng cho phần lớn query.

Evaluation Dataset

Schema:
- query: input để chạy pipeline
- ground-truth-answer: để đánh giá answer correctness
- relevant_document_ids/relevant_chunk_ids: để đánh giá retrieval recall/precision độc lập với chất lượng generation - đây là field bắt buộc để tách lỗi retrieval khỏi lỗi generation
- category: phân nhóm để biết pipeline yếu ơ loại câu hỏi nào 
- difficulty: theo dõi cải điện có đồng đều không, hay chỉ ăn điểm dễ
- language: bắt buộc vì kb + user có cả việt/anh
- expected_source: biết câu trả lời nên đến từ loại tài liệu nào (doc/FAQ/ticket) - hữu ích khi debug trieval chọn sai loại nguồn.

Part 6 - Retrieval Metrics

Recall@K
Công thức: (số relevant document xuất hiện trong top-K) / (tổng số relevant document có trong ground-truth)
Intuition: trong số tài liệu đúng cần tìm, hệ thống có tìm ra được không (bất kể thứ hạng) - đo khả năng không bỏ sót.
Ví dụ: ground-truth có 3 chunk liên quan, hệ thống trả về top-10 và chứa 2/3 chunk đó -> Recall@10 = 66.7%

Precision@K
Công thức: (số relevant document trong top-K) / K
Intuition: trong số K tài liệu hệ thống trả về, bao nhiêu % thực sự liên quan
Ví dụ: top-5 trả về, trong đó 2 chunk thực sự liên quan -> Precision@5 = 40%

MRR (Mean Reciprocal Rank)
Tại sao thứ hạng quan trọng? Vì context budget có giới hạn (Part 3) - nếu chunk đúng nằm ở rank 1, gần như chắc chắn được đưa vào context; nếu nằm ở rank 5, có thể bị cắt bỏ nếu top-N cuối cùng chỉ lấy 3.

Công thức: MRR = (1/N) * sigma (1/rank của relevant document đầu tiên) trên N query
Ví dụ:
query 1: relevant document ở rank 1 -> reciprocal rank = 1/1 = 1.0
query 2: relevant document ở rank 2 -> reciprocal rank = 1/5 = 0.2
MRR trung bình 2 query = (1.0 + 2.0) / 2 = 0.6

MRR phạt nặng trường hợp document đúng "có mặt nhưng xếp hạng thấp" - query 2 dù về mặt Recall@10 vấn tính là tìm thấy, nhưng MRR phản ánh rõ ràng nó kém hơn nhiều so với Query 1 vì user/LLM ít có khả năng thực sự chạm tới nó nếu context bị giới hạn. 

Part 7 - Generation Evaluation
Answer Correctness: Câu trả lời cuối cùng có đúng về mặt nội dung so với ground_truth_answer hay không - đánh giá outcome, không quan tâm quá trình

Faithfulnes (Groundedness): Câu trả lời có chỉ dựa trên thông tin trong context đã cung cấp không, hay LLM tự thêm thông tin từ kiến thức train sẵn (có thể đúng ngẫu nhiên hoặc hoàn toàn bia) - đánh giá quá trình, độc lập với việc câu trả lời cuối có dúng hau không.

